# Convert video to gif

## Import modules

In [1]:
# import internal modules
from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union, Callable
from pathlib import Path
from datetime import date

# import 3rd-party modules
import cv2
from imageio import get_writer
from pygifsicle import optimize

# import local modules
# from core.utils.renderer.get_resize_interpolation import get_interpolation
from core.utils.renderer.resizer import resize_with_crop, resize_with_pad

ModuleNotFoundError: No module named 'pygifsicle'

In [2]:
import numpy as np

In [4]:
video_path = Path("/Users/derrickvanfrausum/Desktop/vrt/db/P1855222-edited_dolly-zoom-in.mp4")
video_path.parent

PosixPath('/Users/derrickvanfrausum/Desktop/vrt/db')

## Read video & convert to gif

In [8]:
rotate_90 = -1
resize_fct = resize_with_pad

# set output shape
out_height, out_width, out_channel = 1920, 1080, 3
# out_height, out_width, out_channel = video_height, video_width, 3

# get current date
today = date.today().strftime("%Y%m%d")

# set input & output video path
video_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/tuto_origami_anim0001-0300.mp4")
out_path = video_path.parent / f"{video_path.stem}_{today}.gif"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_n_frames = video_cap.get(cv2.CAP_PROP_FRAME_COUNT)
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_n_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

writer_mode:str='I'
nb_frames = 30

# get frame indexes
frame_idxs = np.linspace(3, video_cap.get(cv2.CAP_PROP_FRAME_COUNT) - 1, num=nb_frames, dtype=np.uint)

# write gif within context manager
with get_writer(out_path, mode=writer_mode) as writer:

    # iterate over frame indexes
    for frame_idx in frame_idxs:
        # set frame position to the index
        video_cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

        # read video stream
        ret, frame = video_cap.read()

        if not ret:
            print(f"frame {frame_idx} is empty")
            break

        # convert image to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # rotate & resize frame if asked
        if rotate_90 is not None:
            frame = np.rot90(frame, rotate_90)
        
        if resize_fct is not None:
            frame = resize_fct(frame, ref_img_shape=(out_height, out_width, out_channel))

        # write output frame
        writer.append_data(frame)

# release video stream
video_cap.release()

# optimize gif to reduce size
optimize(out_path)

number of frames = 300.0
fps = 24.0
video width = 1920
video height = 1080


In [13]:
frame_idxs

array([  0,   7,  15,  23,  30,  38,  46,  53,  61,  69,  77,  84,  92,
       100, 107, 115, 123, 131, 138, 146, 154, 161, 169, 177, 185, 192,
       200, 208, 215, 223, 231, 239, 246, 254, 262, 269, 277, 285, 293,
       300, 308, 316, 323, 331, 339, 347, 354, 362, 370, 377, 385, 393,
       401, 408, 416, 424, 431, 439, 447, 455], dtype=uint64)

In [1]:
!pip install moviepy

     |████████████████████████████████| 388 kB 4.9 MB/s eta 0:00:01
  Using cached tqdm-4.65.0-py3-none-any.whl (77 kB)
  Using cached requests-2.31.0-py3-none-any.whl (62 kB)
     |████████████████████████████████| 313 kB 9.3 MB/s eta 0:00:01
     |████████████████████████████████| 22.5 MB 9.2 MB/s eta 0:00:01     |████████████████████████████    | 19.8 MB 9.2 MB/s eta 0:00:01
     |████████████████████████████████| 3.4 MB 12.2 MB/s eta 0:00:01
     |████████████████████████████████| 123 kB 7.7 MB/s eta 0:00:01
  Using cached idna-3.4-py3-none-any.whl (61 kB)
     |████████████████████████████████| 122 kB 7.9 MB/s eta 0:00:01
  Created wheel for moviepy: filename=moviepy-1.0.3-py3-none-any.whl size=110726 sha256=8e43ca9c27467d0bc976fda96b293a253d408be3ec926aee6543ada0a6d22994
  Stored in directory: /Users/derrickvanfrausum/Library/Caches/pip/wheels/56/dc/2b/9cd600d483c04af3353d66623056fc03faed76b7518faae4df
Successfully built moviepy
  Attempting uninstall: pillow
    Found existing i

In [3]:
from moviepy.editor import VideoFileClip

def video_to_gif(video_file, gif_file, start_time=0, end_time=None, fps=10):
    """Convert video file to GIF.
    
    Parameters:
    video_file (str): Input video file.
    gif_file (str): Output GIF file.
    start_time (int, optional): Start time of the section of the video to convert. Default is 0.
    end_time (int, optional): End time of the section of the video to convert. If None, converts the whole video. Default is None.
    fps (int, optional): Frames per second in the output GIF. Default is 10.
    """
    
    # Load video
    clip = VideoFileClip(video_file)
    
    # If end time is not specified, use the duration of the clip
    if end_time is None:
        end_time = clip.duration
    
    # Cut the section of the video
    clip = clip.subclip(start_time, end_time)
    
    # Convert to GIF
    clip.to_gif(gif_file, fps=fps)


In [6]:
video_to_gif("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.mp4", "/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.gif", start_time=0, end_time=None, fps=3)

MoviePy - Building file /Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.gif with imageio.


In [12]:
def video_to_gif(video_file, gif_file, start_time=0, end_time=None, fps=10, resize=None, crop=None):
    """Convert video file to GIF.
    
    Parameters:
    video_file (str): Input video file.
    gif_file (str): Output GIF file.
    start_time (int, optional): Start time of the section of the video to convert. Default is 0.
    end_time (int, optional): End time of the section of the video to convert. If None, converts the whole video. Default is None.
    fps (int, optional): Frames per second in the output GIF. Default is 10.
    resize (tuple, optional): New size for the frames as (width, height). If None, keeps the original size. Default is None.
    crop (tuple, optional): Crop rectangle as (x1, y1, x2, y2). If None, no cropping is applied. Default is None.
    """
    
    # Load video
    clip = VideoFileClip(video_file)
    
    # If end time is not specified, use the duration of the clip
    if end_time is None:
        end_time = clip.duration
    
    # Cut the section of the video
    clip = clip.subclip(start_time, end_time)
    
    # Resize frames if requested
    if resize is not None:
        clip = clip.resize(resize)
    
    # Crop frames if requested
    if crop is not None:
        clip = clip.crop(*crop)
    
    # Convert to GIF
    clip.to_gif(gif_file, fps=fps)


# Call the function with your video file
video_to_gif("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.mp4", "/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.gif", 
             start_time=0, end_time=None, fps=4, resize=(1920//2,1080//2),
             crop=(210, 0, 750, 540)
             )


MoviePy - Building file /Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/3d/renders/grease_pencil/0000-0120.gif with imageio.


In [11]:
1920//2,1080//2

(960, 540)